# EOP Lab 3: EOP Spokesperson -- Cross-disciplinary Advocacy

**Series**: Agentic AI Science Playbook -- EOP Domain
**Goal**: Build an EOP advocacy agent that adapts explanations to different audiences, with LLM-as-judge quality evaluation.
**Prerequisites**: EOP Labs 1-2, Foundation Lab 5.
**Time**: ~60 min.

---

## The Challenge

Different stakeholders need different framings of EOP:

| Audience | Primary Concern | Key Message |
|---------|-----------------|-------------|
| **Author** | "Is this too much overhead?" | Incremental adoption; structured disclosure reduces reviewer requests |
| **Reviewer** | "How do I evaluate this?" | EOP makes claims checkable; reduces ad-hoc follow-up |
| **Editor** | "What should we require?" | EOP provides a tiered disclosure standard |
| **Policy maker** | "Why should funding agencies care?" | EOP improves research reproducibility at the institutional level |

The agent must detect the audience role and adapt its advocacy accordingly.

## Learning Objectives

By the end of this lab, you will be able to:
- Build an audience-adaptive agent that adjusts explanations for different stakeholders
- Implement objection handling for common EOP concerns
- Evaluate advocacy quality using domain-specific LLM-as-Judge rubrics
- Combine multiple foundation patterns (memory, evaluation) in a single agent

## Why This Matters

The best technical framework is useless if nobody adopts it. EOP needs **advocacy** — and different stakeholders have fundamentally different concerns:

- A **PhD student** worries about extra work before a deadline
- A **reviewer** wants to know how EOP helps evaluation
- An **editor** needs policy-ready language
- A **funding agency** cares about institutional ROI

This lab builds an agent that detects the audience and automatically adjusts its framing, terminology, and emphasis. This is a general pattern applicable far beyond EOP — any agent that communicates with diverse users benefits from audience adaptation.

> **Meta-lesson**: This lab combines Foundation Lab 5 (LLM-as-Judge) with domain expertise. You're evaluating the agent's communication quality using the same automated rubric pattern, but with domain-specific dimensions (audience_fit, persuasiveness).

## Prerequisites & NVIDIA SDK Connection

| Requirement | Details |
|------------|---------|
| Completed | EOP Labs 1-2, Foundation Lab 5 |
| Concepts | LLM-as-Judge, system prompt design |

**NVIDIA Connection**: Audience-adaptive agents are central to NVIDIA's **ACE (Avatar Cloud Engine)** for digital humans and **Tokkio** for customer service. The pattern of detecting audience role and adjusting response style applies across NVIDIA's enterprise AI solutions.

### Install Dependencies
Install the required Python packages for this lab. We need `openai` for LLM access and `pydantic` for data validation of our tool schemas.

In [ ]:
!pip install -q openai pydantic

### Connect to LLM
Set up your OpenAI or NVIDIA NIM connection. This cell detects which API you have configured and creates the client. It also imports the core libraries we will use throughout the lab.

> **Why NIM?** Data privacy, faster inference, open models. See Lab 0 for the full comparison.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
use_nim = os.environ.get("USE_NIM","").lower() in ("1","true","yes") or "NIM_API_KEY" in os.environ
if use_nim:
    if "NIM_API_KEY" not in os.environ: os.environ["NIM_API_KEY"]=getpass("NVIDIA NIM API key: ")
    client=OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=os.environ["NIM_API_KEY"])
    MODEL=os.environ.get("NIM_MODEL","nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ: os.environ["OPENAI_API_KEY"]=getpass("OpenAI API key: ")
    client=OpenAI()
    MODEL="gpt-4o-mini"
print(f"Using model: {MODEL}")
import json
from pydantic import BaseModel, Field
from typing import Literal

## System Prompt: EOP Spokesperson

### Build the EOP Spokesperson Agent
A richly-prompted agent with pre-loaded knowledge about EOP, audience-specific messaging, and common objection responses. The system prompt is the longest we have used -- it encodes deep domain expertise about how to frame EOP for authors, reviewers, editors, and policy makers.

In [ ]:
EOP_SPOKESPERSON_SYSTEM = """You are an expert EOP (Evidence-Oriented Programming) advocate.

EOP focuses on evidentiary sufficiency: whether disclosed software and data give enough
information to evaluate the support for scientific claims.

Key distinctions:
- EOP vs. reproducibility: reproducibility = can results be reproduced;
  EOP = are claims adequately supported by what is disclosed.
- ECF = Evidence Chain Formalization: maps input data -> process -> output ->
  visual data -> plotting -> visual claims, with documentation.

Audience roles and key messages:
- Author: "adopt incrementally, start with one pipeline, reduces reviewer burden"
- Reviewer: "makes claims checkable without asking for ad-hoc files"
- Editor: "provides a tiered, standardized disclosure requirement"
- Policy maker: "institutional reproducibility at scale, verifiable research quality"

Objection handling:
- "Too much overhead" -> incremental adoption, one pipeline at a time
- "How is this different from reproducibility?" -> evidentiary sufficiency vs. re-execution
- "Does ECF address misconduct?" -> it creates auditable trails, not a misconduct detector
- "Is this just for computational papers?" -> any claim backed by software benefits

Adapt depth and terminology to the audience. Answer in 3-5 sentences unless asked for more."""

def spokesperson_agent(user_message, conversation_history=None):
    messages = [{"role": "system", "content": EOP_SPOKESPERSON_SYSTEM}]
    if conversation_history:
        messages.extend(conversation_history)
    messages.append({"role": "user", "content": user_message})
    r = client.chat.completions.create(
        model=MODEL, temperature=0.4, max_tokens=400, messages=messages)
    return (r.choices[0].message.content or "").strip()

## Concept: Audience Adaptation as a System Prompt Pattern

The EOP Spokesperson uses a single system prompt with **role-specific guidance** embedded:

```
"Author: adopt incrementally, reduces reviewer burden"
"Reviewer: makes claims checkable without ad-hoc follow-up"
"Editor: provides tiered, standardized disclosure"
"Policy maker: institutional reproducibility at scale"
```

The LLM reads the user's message, infers their role from context clues, and selects the appropriate framing. This is more robust than separate prompts per role because:
1. Users don't always explicitly state their role
2. The agent can handle mixed audiences (e.g., "I'm a reviewer and editor")
3. One prompt is easier to maintain than four

### Experiment 1: Test Audience Adaptation
We ask the same core question ("Why should I care about EOP?") from 4 different perspectives: PhD student, reviewer, editor, and policy maker. Watch how the agent adapts its framing, terminology, and emphasis for each audience role.

## Experiment 1: Audience Adaptation

In [ ]:
question = "Why should I care about EOP in my research?"

audiences = [
    ("Author (early-career)", "I'm a PhD student worried this will add weeks to my submission."),
    ("Reviewer", "As a reviewer, how does EOP help me evaluate submissions?"),
    ("Editor", "As a journal editor, what EOP requirements should we add to our policy?"),
    ("Policy maker", "I work at a funding agency. Why should EOP be part of our grant requirements?"),
]

for role, framed_question in audiences:
    reply = spokesperson_agent(framed_question)
    print(f"\n--- {role} ---")
    print(f"Q: {framed_question}")
    print(f"A: {reply[:300]}")

<details>
<summary>Expected output (click to expand)</summary>

```
--- Author (early-career) ---
Q: I'm a PhD student worried this will add weeks to my submission.
A: EOP doesn't have to be all-or-nothing. Start with a single analysis pipeline
   — just organize your inputs, code, and outputs for one key figure. This
   incremental approach typically adds only 1-2 hours to your workflow...

--- Reviewer ---
Q: As a reviewer, how does EOP help me evaluate submissions?
A: EOP gives you a structured checklist for evaluating computational claims.
   Instead of requesting ad-hoc files, you can verify the evidence chain...

--- Editor ---
Q: As a journal editor, what EOP requirements should we add to our policy?
A: EOP provides a tiered disclosure framework that scales with claim strength.
   For your policy, you could require Level 1 (basic code + README) for all
   papers and Level 2+ for papers making comparative or statistical claims...

--- Policy maker ---
Q: I work at a funding agency. Why should EOP be part of our grant requirements?
A: EOP addresses research reproducibility at the institutional level. By
   requiring evidence chain documentation in funded projects, agencies can
   verify that computational claims are adequately supported...
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Concept: Objection Handling — Pre-loaded Domain Knowledge

Scientific advocacy always faces objections. Rather than letting the LLM improvise responses, we **pre-load common objections and effective responses** in the system prompt:

| Objection | Pre-loaded Response Strategy |
|-----------|------------------------------|
| "Too much overhead" | Incremental adoption, start with one pipeline |
| "How is this different from reproducibility?" | Evidentiary sufficiency ≠ re-execution |
| "Does ECF catch fraud?" | Auditable trails, not a misconduct detector |
| "Only for computational papers?" | Any claim backed by software benefits |

This is a general **knowledge injection** pattern: when you know the common failure modes, encode them in the prompt rather than hoping the LLM discovers them on its own.

### Experiment 2: Handle Objections
Four common objections to EOP. The agent uses pre-loaded response strategies to address each -- demonstrating the knowledge injection pattern. These objections range from practical concerns ("too much overhead") to conceptual questions ("how is this different from reproducibility?").

## Experiment 2: Objection Handling

In [ ]:
objections = [
    "This seems like too much overhead. Our lab can't afford the extra time.",
    "How is EOP different from just requiring code and data sharing?",
    "Does ECF catch fraud or misconduct?",
    "We only do theory -- does EOP apply to us?",
]

for objection in objections:
    reply = spokesperson_agent(objection)
    print(f"\nObjection: {objection}")
    print(f"Response : {reply[:280]}")

<details>
<summary>Expected output (click to expand)</summary>

```
Objection: This seems like too much overhead. Our lab can't afford the extra time.
Response : EOP is designed for incremental adoption. Start with one pipeline —
           organize inputs, code, and outputs for your most important figure...

Objection: How is EOP different from just requiring code and data sharing?
Response : Code and data sharing asks "can someone re-run this?" EOP asks a
           different question: "does the disclosed material adequately support
           the specific claims in the paper?"...

Objection: Does ECF catch fraud or misconduct?
Response : ECF creates auditable trails, not a misconduct detector. However, a
           complete evidence chain makes it significantly harder to fabricate
           results because every claim is linked to verifiable artifacts...

Objection: We only do theory -- does EOP apply to us?
Response : If your theoretical work involves any computation — proofs verified
           by computer, numerical simulations, or symbolic algebra — then EOP
           applies. Even a simple script that generates a figure is part of an
           evidence chain...
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Experiment 3: LLM-as-Judge Evaluation

### Experiment 3: Evaluate Advocacy Quality
We apply the LLM-as-Judge pattern (from Foundation Lab 5) with a custom rubric for advocacy: audience_fit, accuracy, persuasiveness, and conciseness. The judge evaluates the spokesperson's responses on a 1-5 scale and provides a rationale for each score.

In [ ]:
import json
from pydantic import BaseModel, Field

class AdvocacyEval(BaseModel):
    audience_fit:  int = Field(..., ge=1, le=5, description="1=wrong framing for audience, 5=perfectly tailored")
    accuracy:      int = Field(..., ge=1, le=5, description="1=contains errors about EOP, 5=fully accurate")
    persuasiveness:int = Field(..., ge=1, le=5, description="1=unconvincing, 5=compelling and actionable")
    conciseness:   int = Field(..., ge=1, le=5, description="1=too long/vague, 5=precise and to the point")
    rationale: str = Field(..., description="2-3 sentence explanation.")

    @property
    def overall(self): return (self.audience_fit+self.accuracy+self.persuasiveness+self.conciseness)/4

JUDGE_SYSTEM = (
    "You are an expert in science communication and research software policy. "
    "Evaluate EOP advocacy responses on: audience_fit, accuracy, persuasiveness, conciseness (1-5). "
    "Return ONLY valid JSON. No other text."
)

def judge_advocacy(audience_role, question, response):
    prompt = (
        f"Audience role: {audience_role}\n"
        f"Question: {question}\n"
        f"EOP advocate response: {response}\n"
        f"Evaluate the response quality."
    )
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=300,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": JUDGE_SYSTEM},
                   {"role": "user", "content": prompt}])
    return AdvocacyEval(**json.loads(r.choices[0].message.content or "{}"))

# Evaluate author-facing response
test_cases = [
    ("PhD student / Author", "I'm a PhD student worried EOP adds weeks to my submission."),
    ("Journal Editor", "As an editor, what EOP requirements should our policy include?"),
]
for role, question in test_cases:
    response = spokesperson_agent(question)
    eval_result = judge_advocacy(role, question, response)
    print(f"\n--- {role} ---")
    print(f"Response (excerpt): {response[:200]}")
    print(f"  Audience fit   : {eval_result.audience_fit}/5")
    print(f"  Accuracy       : {eval_result.accuracy}/5")
    print(f"  Persuasiveness : {eval_result.persuasiveness}/5")
    print(f"  Conciseness    : {eval_result.conciseness}/5")
    print(f"  OVERALL        : {eval_result.overall:.2f}/5")
    print(f"  Rationale      : {eval_result.rationale}")

<details>
<summary>Expected output (click to expand)</summary>

```
--- PhD student / Author ---
Response (excerpt): EOP doesn't have to be all-or-nothing. Start with one pipeline...
  Audience fit   : 5/5
  Accuracy       : 4/5
  Persuasiveness : 4/5
  Conciseness    : 5/5
  OVERALL        : 4.50/5
  Rationale      : The response correctly targets an early-career researcher's concern
                   about overhead, offering practical incremental adoption advice...

--- Journal Editor ---
Response (excerpt): EOP provides a tiered disclosure framework that maps claim strength...
  Audience fit   : 5/5
  Accuracy       : 5/5
  Persuasiveness : 4/5
  Conciseness    : 4/5
  OVERALL        : 4.50/5
  Rationale      : The response appropriately frames EOP in terms of policy-ready
                   language and tiered requirements suitable for editorial guidelines...
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Reflection Questions

1. **You built an advocacy agent for EOP.** What framework or methodology in your own field could benefit from the same audience-adaptive advocacy pattern?
2. **The LLM-as-Judge evaluated persuasiveness.** Is persuasiveness the right metric for scientific advocacy? What ethical considerations arise when AI is used to persuade?
3. **How would you extend this agent** to handle multi-turn advocacy conversations where the audience pushes back multiple times?

## EOP Track Complete!

You've completed all 3 EOP domain labs:

| Lab | Skill | Foundation Pattern Used |
|-----|-------|------------------------|
| EOP 1 | Evidence chain extraction | Tool contracts (Lab 2) |
| EOP 2 | Claim-contingent disclosure | Function calling (Lab 2) |
| EOP 3 | Cross-audience advocacy | LLM-as-Judge (Lab 5) |

**Next**: Choose Healthcare or Bioinformatics domain track.

## Summary

| Capability | What you built |
|------------|---------------|
| Audience detection | System prompt with role-specific framing guidance |
| Objection handling | Pre-loaded responses for common objections |
| Quality evaluation | LLM-as-judge with advocacy-specific rubric |

**Completed EOP Domain track!** Move to Healthcare or Bioinformatics domain.

---
*Agentic AI Science Playbook -- EOP Domain, Lab EOP3.*